# 17a_generate_mel_nv_clean_qualitative_and_cross_eval

Clean MEL/NV qualitative evaluation and checkpoint cross evaluation.

This notebook does three things:

1. Creates a small **clean** MEL/NV qualitative CSV from the clean test split.
2. Generates qualitative CAM/FinerCAM panels for the **Clean CE** checkpoint on clean images.
3. Runs a cross evaluation matrix to check whether the cued models depend on the cue:
   - Clean CE on clean test
   - Clean CE on cued test
   - Cue CE on clean test
   - Cue CE on cued test
   - Cue HA on clean test
   - Cue HA on cued test

Run this notebook from the repo `notebooks/` folder.


In [1]:
from pathlib import Path
import json
import subprocess
import shlex
import pandas as pd

QUAL_SEED = 42

REPO_ROOT = Path("..").resolve()
HAM_ROOT = REPO_ROOT / "data" / "HAM10000"
SYN_ROOT = HAM_ROOT / "synthetic_cue" / f"mel_nv_fixed_center_seed{QUAL_SEED}"

CLEAN_CSV = SYN_ROOT / "csv" / "ham_mel_nv_clean.csv"
CUE_CSV = SYN_ROOT / "csv" / f"ham_mel_nv_cue_on_mel_fixed_center_seed{QUAL_SEED}.csv"

CLEAN_QUAL_CSV = SYN_ROOT / "csv" / f"ham_mel_nv_clean_qualitative_10_seed{QUAL_SEED}.csv"

QUAL_ROOT = REPO_ROOT / "outputs" / f"qual_clean_mel_nv_seed{QUAL_SEED}"
QUAL_ROOT.mkdir(parents=True, exist_ok=True)

IMG_DIR = HAM_ROOT
MASK_ROOT = HAM_ROOT

print("REPO_ROOT:", REPO_ROOT)
print("CLEAN_CSV:", CLEAN_CSV)
print("CUE_CSV:", CUE_CSV)
print("QUAL_ROOT:", QUAL_ROOT)

REPO_ROOT: /Users/choekyelnyungmartsang/Developer/master-thesis
CLEAN_CSV: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue/mel_nv_fixed_center_seed42/csv/ham_mel_nv_clean.csv
CUE_CSV: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue/mel_nv_fixed_center_seed42/csv/ham_mel_nv_cue_on_mel_fixed_center_seed42.csv
QUAL_ROOT: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/qual_clean_mel_nv_seed42


## 1. Create clean qualitative CSV

Select 5 MEL and 5 NV test images from the clean MEL/NV CSV. These images should not contain the synthetic green cue.


In [2]:
df = pd.read_csv(CLEAN_CSV)

required_cols = ["image_id", "gt_label", "split", "image_rel_path", "mask_rel_path", "mask_found"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in {CLEAN_CSV}: {missing}")

test_df = df[df["split"] == "test"].copy()
mel = test_df[test_df["gt_label"] == "MEL"].sample(n=5, random_state=QUAL_SEED)
nv = test_df[test_df["gt_label"] == "NV"].sample(n=5, random_state=QUAL_SEED)

qual_df = pd.concat([mel, nv], axis=0).copy()
qual_df["order"] = qual_df["gt_label"].map({"MEL": 0, "NV": 1})
qual_df = qual_df.sort_values(["order", "image_id"]).drop(columns=["order"])

CLEAN_QUAL_CSV.parent.mkdir(parents=True, exist_ok=True)
qual_df.to_csv(CLEAN_QUAL_CSV, index=False)

print("Saved:", CLEAN_QUAL_CSV)
print("Rows:", len(qual_df))
print("Counts:")
print(qual_df["gt_label"].value_counts())

display(qual_df[["image_id", "gt_label", "split", "image_rel_path", "mask_rel_path", "mask_found"]])

Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue/mel_nv_fixed_center_seed42/csv/ham_mel_nv_clean_qualitative_10_seed42.csv
Rows: 10
Counts:
gt_label
MEL    5
NV     5
Name: count, dtype: int64


,image_id,gt_label,split,image_rel_path,mask_rel_path,mask_found
0,ISIC_0024459,MEL,test,images/ISIC_0024459.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,1
4,ISIC_0024756,MEL,test,images/ISIC_0024756.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,1
22,ISIC_0026993,MEL,test,images/ISIC_0026993.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,1
49,ISIC_0030798,MEL,test,images/ISIC_0030798.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,1
54,ISIC_0031408,MEL,test,images/ISIC_0031408.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,1
70,ISIC_0024416,NV,test,images/ISIC_0024416.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,1
74,ISIC_0024874,NV,test,images/ISIC_0024874.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,1
92,ISIC_0027312,NV,test,images/ISIC_0027312.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,1
119,ISIC_0029892,NV,test,images/ISIC_0029892.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,1
124,ISIC_0030763,NV,test,images/ISIC_0030763.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,1


## 2. Define checkpoints

Adjust the checkpoint paths if your local filenames differ.


In [3]:
SIZE = "small"
CHECKPOINTS = {
    "Clean CE": REPO_ROOT / "external" / f"checkpoints2_{SIZE}" / "checkpoint-best-clean.pth",
    "Cue CE": REPO_ROOT / "external" / f"checkpoints2_{SIZE}" / "checkpoint-best-cue.pth",
    "Cue HA": REPO_ROOT / "external" / f"checkpoints2_{SIZE}" / "checkpoint-best-cue-ha.pth",
}

EXPERIMENTS_CLEAN = {
    "Clean CE": {
        "checkpoint": CHECKPOINTS["Clean CE"],
        "checkpoint_model_type": "panderm",
        "use_seg_gate": False,
        "out_dir": QUAL_ROOT / "cam_clean_ce_on_clean_test",
    },
}

for name, ckpt in CHECKPOINTS.items():
    print(name, "->", ckpt)
    if not ckpt.exists():
        print("  [WARN] missing checkpoint. Adjust path above.")

Clean CE -> /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_small/checkpoint-best-clean.pth
Cue CE -> /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_small/checkpoint-best-cue.pth
Cue HA -> /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_small/checkpoint-best-cue-ha.pth


## 3. Helper functions

In [4]:
def run_command(cmd: list[str], dry_run: bool = False):
    print("\n" + "=" * 100)
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 100)
    if dry_run:
        return
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def generate_qualitative_cams(experiments: dict, csv_path: Path, img_dir: Path, num_samples: int, dry_run: bool = False):
    for exp_name, cfg in experiments.items():
        out_dir = cfg["out_dir"]
        out_dir.mkdir(parents=True, exist_ok=True)

        panel_items = "rgb_gt_mask,gradcam_a,gradcam_b,map_diff,finercam"

        cmd = [
            "python", "-m", "scripts.generate_finer_cam_panderm",
            "--csv", str(csv_path),
            "--image_col", "image_rel_path",
            "--img_dir", str(img_dir),
            "--gt_col", "gt_label",
            "--checkpoint", str(cfg["checkpoint"]),
            "--checkpoint_model_type", cfg["checkpoint_model_type"],
            "--class_names", "MEL,NV",
            "--out_dir", str(out_dir),
            "--num_samples", str(num_samples),
            "--method", "finercam",
            "--compare_mode", "gt_pair",
            "--A", "MEL",
            "--B", "NV",
            "--topk_compare", "1",
            "--alpha", "0.8",
            "--panel_items", panel_items,
            "--mask_root", str(MASK_ROOT),
            "--mask_col", "mask_rel_path",
            "--save_json",
            "--target_block_index", str(-4),
        ]

        print(f"\nRunning clean CAM generation: {exp_name}")
        run_command(cmd, dry_run=dry_run)


def build_qualitative_pdf(csv_path: Path, experiments_json_path: Path, out_pdf: Path, num_samples: int = 10, dry_run: bool = False):
    out_pdf.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python", "-m", "scripts.make_qualitative_comparison_pdf",
        "--csv", str(csv_path),
        "--image_col", "image_rel_path",
        "--gt_col", "gt_label",
        "--out_pdf", str(out_pdf),
        "--experiments_json_path", str(experiments_json_path),
        "--num_samples", str(num_samples),
        "--missing_policy", "placeholder",
    ]

    run_command(cmd, dry_run=dry_run)


def run_eval(checkpoint: Path, csv_path: Path, output_name: str, dry_run: bool = False):
    out_dir = REPO_ROOT / "outputs" / output_name
    cmd = [
        "python", "-m", "scripts.run_panderm_full_finetune_ha",
        "--panderm-classification-dir", "external/PanDerm/classification",
        "--csv-path", str(csv_path),
        "--root-path", str(HAM_ROOT),
        "--pretrained-checkpoint", "external/weights/panderm_bb_data6_checkpoint-499.pth",
        "--output-dir", str(out_dir),
        "--model", "PanDerm_Base_FT",
        "--nb-classes", "2",
        "--batch-size", "64",
        "--epochs", "1",
        "--lr", "5e-5",
        "--weight-decay", "0.05",
        "--warmup-epochs", "0",
        "--layer-decay", "0.65",
        "--drop-path", "0.2",
        "--update-freq", "1",
        "--seed", "0",
        "--num-workers", "4",
        "--monitor", "recall",
        "--weights",
        "--image-key", "image_rel_path",
        "--mask-key", "mask_rel_path",
        "--ha-lambda", "0.0",
        "--dal-lambda", "0.0",
        "--init-checkpoint", str(checkpoint),
        "--wandb-mode", "disabled",
        "--eval-only",
        "--disable-color-jitter",
        "--disable-amp",
        "--debug-batches", "0",
        "--device", "cpu",
    ]
    run_command(cmd, dry_run=dry_run)

## 4. Generate clean qualitative CAM panels

In [5]:
generate_qualitative_cams(
    experiments=EXPERIMENTS_CLEAN,
    csv_path=CLEAN_QUAL_CSV,
    img_dir=IMG_DIR,
    num_samples=10,
    dry_run=False,
)


Running clean CAM generation: Clean CE

python -m scripts.generate_finer_cam_panderm --csv /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue/mel_nv_fixed_center_seed42/csv/ham_mel_nv_clean_qualitative_10_seed42.csv --image_col image_rel_path --img_dir /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000 --gt_col gt_label --checkpoint /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_small/checkpoint-best-clean.pth --checkpoint_model_type panderm --class_names MEL,NV --out_dir /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/qual_clean_mel_nv_seed42/cam_clean_ce_on_clean_test --num_samples 10 --method finercam --compare_mode gt_pair --A MEL --B NV --topk_compare 1 --alpha 0.8 --panel_items rgb_gt_mask,gradcam_a,gradcam_b,map_diff,finercam --mask_root /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000 --mask_col mask_rel_path --save_json --target_block_index -4
[info] Loaded PanDerm Base FT

## 5. Build clean qualitative PDF

In [6]:
CONFIG_DIR = REPO_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

clean_pdf_config = [
    {"name": name, "folder": str(cfg["out_dir"].relative_to(REPO_ROOT))}
    for name, cfg in EXPERIMENTS_CLEAN.items()
]

CLEAN_JSON = CONFIG_DIR / f"qualitative_clean_mel_nv_seed{QUAL_SEED}.json"
CLEAN_JSON.write_text(json.dumps(clean_pdf_config, indent=2))

print("Saved:", CLEAN_JSON)
print(json.dumps(clean_pdf_config, indent=2))

build_qualitative_pdf(
    csv_path=CLEAN_QUAL_CSV,
    experiments_json_path=CLEAN_JSON,
    out_pdf=QUAL_ROOT / f"qualitative_clean_mel_nv_seed{QUAL_SEED}.pdf",
    num_samples=10,
    dry_run=False,
)

Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/configs/qualitative_clean_mel_nv_seed42.json
[
  {
    "name": "Clean CE",
    "folder": "outputs/qual_clean_mel_nv_seed42/cam_clean_ce_on_clean_test"
  }
]

python -m scripts.make_qualitative_comparison_pdf --csv /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue/mel_nv_fixed_center_seed42/csv/ham_mel_nv_clean_qualitative_10_seed42.csv --image_col image_rel_path --gt_col gt_label --out_pdf /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/qual_clean_mel_nv_seed42/qualitative_clean_mel_nv_seed42.pdf --experiments_json_path /Users/choekyelnyungmartsang/Developer/master-thesis/configs/qualitative_clean_mel_nv_seed42.json --num_samples 10 --missing_policy placeholder
Saved PDF: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/qual_clean_mel_nv_seed42/qualitative_clean_mel_nv_seed42.pdf
Pages written: 10


## 6. Cross evaluation matrix

This checks cue reliance. Important comparison:

- Cue CE on cued test vs Cue CE on clean test
- Cue HA on cued test vs Cue HA on clean test

If the cued checkpoints rely strongly on the green cue, performance should drop on the clean test CSV.


In [7]:
# EVAL_JOBS = [
#     ("clean_ce_on_clean", CHECKPOINTS["Clean CE"], CLEAN_CSV),
#     ("clean_ce_on_cued", CHECKPOINTS["Clean CE"], CUE_CSV),
#     ("cue_ce_on_clean", CHECKPOINTS["Cue CE"], CLEAN_CSV),
#     ("cue_ce_on_cued", CHECKPOINTS["Cue CE"], CUE_CSV),
#     ("cue_ha_on_clean", CHECKPOINTS["Cue HA"], CLEAN_CSV),
#     ("cue_ha_on_cued", CHECKPOINTS["Cue HA"], CUE_CSV),
# ]

# for name, ckpt, csv_path in EVAL_JOBS:
#     print("\nCross eval:", name)
#     run_eval(
#         checkpoint=ckpt,
#         csv_path=csv_path,
#         output_name=f"eval_{name}_seed{QUAL_SEED}",
#         dry_run=False,
#     )

## Interpretation checklist

- Clean CE on clean test is the natural clean baseline.
- Clean CE on cued test is an out of distribution cue insertion check.
- Cue CE / Cue HA on clean test tests whether the cued model still works when the shortcut is removed.
- Cue CE / Cue HA on cued test tests maximum performance with the shortcut present.
